In [1]:
import requests
session = requests.session()
base_url = "https://maziar-temp-project.ir/connect4"

In [ ]:
#TODO Change the emails to your own email and your opponent emain and please go to https://maziarheidari2003.github.io/connect4game and make an account first

# This is the bot player
bot_auth_data = session.post(f"{base_url}/api/v1/auth/login", json={"email":"yalda@gmail.com"}).json()
bot_id = bot_auth_data["id"]
bot_token ="Bearer " + bot_auth_data["access_token"]

# This is the player who should go to the browser after creating and joining the game
opponent_auth_data = session.post(f"{base_url}/api/v1/auth/login", json={"email":"mousavi@gmail.com"}).json()
opponent_id = opponent_auth_data["id"]
opponent_token = "Bearer " + opponent_auth_data["access_token"]


In [ ]:
import time
import random

def bot_make_move(token, game_uuid, opponent_last_move: int | None = None):
    if not hasattr(bot_make_move, "board"):
        bot_make_move.board = [[0] * 7 for _ in range(6)]

    board = bot_make_move.board
    start_time = time.time()

    def get_open_row(b, col):
        for r in range(5, -1, -1):
            if b[r][col] == 0: return r
        return None

    def evaluate_window(window, piece):
        score = 0
        opp_piece = 2 if piece == 1 else 1
        if window.count(piece) == 4: score += 1000
        elif window.count(piece) == 3 and window.count(0) == 1: score += 10
        elif window.count(piece) == 2 and window.count(0) == 2: score += 5
        if window.count(opp_piece) == 3 and window.count(0) == 1: score -= 80
        return score

    def score_position(b, piece):
        score = 0
        center_array = [row[3] for row in b]
        score += center_array.count(piece) * 3

        for r in range(6):
            row_array = b[r]
            for c in range(4):
                score += evaluate_window(row_array[c:c+4], piece)
        for c in range(7):
            col_array = [b[r][c] for r in range(6)]
            for r in range(3):
                score += evaluate_window(col_array[r:r+4], piece)
        for r in range(3):
            for c in range(4):
                window = [b[r+i][c+i] for i in range(4)]
                score += evaluate_window(window, piece)
                window = [b[r+3-i][c+i] for i in range(4)]
                score += evaluate_window(window, piece)
        return score

    def is_terminal(b):
        return score_position(b, 1) > 500 or score_position(b, 2) < -500 or all(b[0][c] != 0 for c in range(7))

    def minimax(b, depth, alpha, beta, maximizing):
        valid_cols = [c for c in [3, 2, 4, 1, 5, 0, 6] if b[0][c] == 0]
        if depth == 0 or is_terminal(b) or (time.time() - start_time > 2.2):
            return None, score_position(b, 1) - score_position(b, 2)

        if maximizing:
            v = -float('inf')
            best_c = random.choice(valid_cols) if valid_cols else 3
            for c in valid_cols:
                r = get_open_row(b, c)
                b[r][c] = 1
                _, score = minimax(b, depth-1, alpha, beta, False)
                b[r][c] = 0
                if score > v:
                    v = score
                    best_c = c
                alpha = max(alpha, v)
                if alpha >= beta: break
            return best_c, v
        else:
            v = float('inf')
            best_c = random.choice(valid_cols) if valid_cols else 3
            for c in valid_cols:
                r = get_open_row(b, c)
                b[r][c] = 2
                _, score = minimax(b, depth-1, alpha, beta, True)
                b[r][c] = 0
                if score < v:
                    v = score
                    best_c = c
                beta = min(beta, v)
                if alpha >= beta: break
            return best_c, v

    if opponent_last_move is not None:
        try:
            c_opp = int(opponent_last_move)
            r_opp = get_open_row(board, c_opp)
            if r_opp is not None: board[r_opp][c_opp] = 2
        except: pass

    col, _ = minimax(board, 3, -float('inf'), float('inf'), True)

    if col is None or board[0][col] != 0:
        col = next(c for c in [3, 2, 4, 1, 5, 0, 6] if board[0][c] == 0)

    r_own = get_open_row(board, col)
    if r_own is not None: board[r_own][col] = 1

    session.patch(f"{base_url}/api/v1/game/make-move",
                  params={"game_uuid": game_uuid, "chosen_column": col},
                  headers={"Authorization": token})

In [ ]:
#Run this to create the game
created_game_uuid = session.post(f"{base_url}/api/v1/game/?bot_or_human=OneSideBot",  headers={"Authorization": bot_token}).json()

In [ ]:
import json, websockets, asyncio

async def human_vs_bot():
    uri = f"wss://maziar-temp-project.ir/connect4/ws/api/v1/game/{created_game_uuid}"

    async with websockets.connect(uri) as ws:
        print("WebSocket connected")
        game = session.patch(f"{base_url}/api/v1/game/join?game_uuid={created_game_uuid}", headers={"Authorization": opponent_token}).json()

        print("You have 20 second to join the game dear human being! Visit https://maziarheidari2003.github.io/connect4game ")

        msg = json.loads(await ws.recv())
        await asyncio.sleep(20)

        while True:
            status = msg["status"]
            if status == "FINISHED":
                print("Game finished. Winner:", msg["winner"])
                break

            current_turn = msg.get("current_turn_id")
            if current_turn == bot_id:
                bot_make_move(bot_token, created_game_uuid, msg.get("last_move"))

            msg = json.loads(await ws.recv())


await human_vs_bot()
